# **Feature visualization библиотекой: torch-lucent и частотная параметризация**

Практика к модулю [«Анализ CNN-based моделей»](https://ai-interpretability.school), урок «Feature Visualization библиотекой: Практика».

Тетрадь [«Feature visualization: попросить сеть нарисовать то, что она ищет»](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/PRACTICE_feature_visualization.ipynb) собирает метод руками на ResNet18: градиентный подъем по входу, сдвиги и штраф за перепады. Третий прием урока — замену параметризации картинки на частотную — она не берет. Здесь мы возьмем библиотеку [torch-lucent](https://github.com/greentfrapp/lucent), PyTorch-порт Lucid, где этот прием включен по умолчанию, и разберем, что делает каждая ее часть:

1. один вызов `render.render_vis` для того же канала 12 слоя `layer4`;
2. что дает частотная параметризация с декорреляцией цветов и что дают преобразования;
3. сравнение с настоящими снимками: активация и перепады;
4. другие цели: нейрон, логит класса, разнообразие;
5. где ломается: зависимость от сида и от размера картинки.

Тетрадь идет на процессоре; весь прогон занимает две-три минуты.

Автор фото кошки: [Paul Hanaoka, Unsplash](https://unsplash.com/photos/w2DsS-ZAP4U), лицензия Unsplash.

Автор фото хрюшки: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), лицензия Pexels.

Автор фото кошки и мопса: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), лицензия Unsplash.

Автор фото велосипедиста: [Tim Foster, Unsplash](https://unsplash.com/photos/wX64_Gjbclg), лицензия Unsplash.

In [ ]:
!pip install -q torch-lucent==0.1.8   # последний релиз — май 2021; проверяйте совместимость с вашим torch

In [ ]:
import io
import urllib.request
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from lucent.optvis import render, param, objectives

warnings.filterwarnings('ignore', category=UserWarning)   # kornia 0.4 и torchvision предупреждают об устаревших аргументах
SIZE = 128        # сторона картинки
STEPS = 256       # шагов оптимизации
SEED = 0

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)

## 1. Один вызов

В lucent цель задается строкой `"<слой>:<канал>"`, где слой — имя модуля, как в `model.named_modules()`, с подчеркиванием вместо точки (`layer4`, `layer3_1_conv2`). `render_vis` сам параметризует картинку, сам добавляет преобразования (сдвиги, масштаб, поворот), сам нормирует вход под ImageNet, растягивает картинку со стороной меньше 224 до 224 и возвращает картинки на шагах из `thresholds` в виде массивов `(1, H, W, 3)` со значениями в $[0,1]$.

Функция `vis` ниже фиксирует сид перед каждым вызовом: иначе стартовый шум и случайные преобразования будут другими при каждом запуске.

In [ ]:
def vis(objective, param_f=None, transforms_=None, seed=SEED, steps=STEPS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    imgs = render.render_vis(model, objective, param_f=param_f, transforms=transforms_, thresholds=(steps,),
                             show_image=False, progress=False)
    return imgs[0]


def show_row(images, titles, size=3):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size + 0.4))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(im[0])
        ax.set_title(t, fontsize=9)
        ax.axis('off')
    plt.show()


lucent_default = vis('layer4:12', param_f=lambda: param.image(SIZE))
show_row([lucent_default], ['lucent, defaults: layer4:12'], size=4)

## 2. Из чего состоит результат

`param.image(SIZE)` по умолчанию включает два приема:

- **частотная параметризация** (`fft=True`): оптимизируются не пиксели, а коэффициенты Фурье картинки, причем высокие частоты масштабированы вниз. Шаг градиента по коэффициентам меняет картинку плавно, и высокочастотный шум, из которого состоит состязательный пример, получает малый вес;
- **декорреляция цветов** (`decorrelate=True`): оптимизируются не каналы RGB, а их некоррелированные комбинации, посчитанные по ImageNet.

Отдельно `render_vis` добавляет случайные **преобразования** на каждом шаге — сдвиги, масштаб, повороты; `transforms_=[]` их выключает.

**Задание 1. Получите четыре картинки для `layer4:12`: пиксельную параметризацию без преобразований (`param.image(SIZE, fft=False, decorrelate=False)` и `transforms_=[]`), пиксельную с преобразованиями по умолчанию, частотную с декорреляцией без преобразований и частотную с декорреляцией с преобразованиями (это `lucent_default`). Нарисуйте их в ряд.**

In [ ]:
# Ваш код здесь

## 3. Сравнение с настоящими снимками

Как в тетради с ручной версией, сравним активацию канала на нарисованных картинках с активацией на четырех настоящих снимках и посчитаем средний перепад между соседними пикселями — меру «шумности». Активацию меряем одинаково для всех картинок и так, как ее видела оптимизация: `render_vis` растягивает картинку со стороной меньше 224 до 224 (билинейно, `align_corners=True`) и нормирует под ImageNet, и мы делаем то же самое со всеми картинками, в том числе со снимками, уменьшенными до 128. Активация — среднее по карте канала 12 выхода `layer4`.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
captured = {}
model.layer4.register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))


def activation(img, channel=12):
    # img — массив (1, H, W, 3) в [0, 1], как возвращает render_vis
    t = torch.tensor(img, dtype=torch.float32).permute(0, 3, 1, 2)
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=True)   # так render_vis растягивает картинки меньше 224
    with torch.no_grad():
        model((t - MEAN) / STD)
    return captured['a'][0, channel].mean().item()


def roughness(img):
    return float(np.abs(np.diff(img, axis=1)).mean() + np.abs(np.diff(img, axis=2)).mean())


FILES = ['data/cat.jpg', 'data/piglet.jpg', 'data/cat_and_dog.jpg', 'assets/tim-foster-w-X64-Gjbclg-unsplash.jpg']
to_arr = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
real = {}
for path in FILES:
    raw = urllib.request.urlopen('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/' + path, timeout=30).read()
    real[path.split('/')[-1]] = to_arr(Image.open(io.BytesIO(raw)).convert('RGB')).permute(1, 2, 0).numpy()[None]

**Задание 2. Для четырех картинок задания 1 и четырех настоящих снимков выведите активацию канала 12 и средний перепад между соседними пикселями. Во сколько раз активация каждой нарисованной картинки выше, чем у лучшего из четырех снимков?**

In [ ]:
# Ваш код здесь

## 4. Другие цели

Строка `"layer4:12"` — сокращение для `objectives.channel("layer4", 12)`. Цели в lucent складываются и умножаются на числа, как функции потерь:

- `objectives.neuron("layer4", 12)` — одна позиция в центре карты канала, а не среднее по карте;
- `"fc:<класс>"` — логит класса; ниже класс 341 ImageNet, `hog` (свинья, как на снимке `piglet.jpg`);
- `objectives.channel(...) - 1e2 * objectives.diversity("layer4")` для батча из нескольких картинок — штраф за похожесть картинок батча друг на друга по матрицам Грама слоя: оптимизация ищет несколько разных способов включить канал.

In [ ]:
neuron_img = vis(objectives.neuron('layer4', 12), param_f=lambda: param.image(SIZE))
class_img = vis('fc:341', param_f=lambda: param.image(SIZE))
show_row([lucent_default, neuron_img, class_img],
         ['channel layer4:12 (mean over map)', 'neuron layer4:12 (center position)', 'class logit fc:341 (hog)'])

diverse = vis(objectives.channel('layer4', 12) - 1e2 * objectives.diversity('layer4'),
              param_f=lambda: param.image(SIZE, batch=4))
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for k, ax in enumerate(axes):
    ax.imshow(diverse[k])
    ax.set_title(f'diversity batch, image {k}: {activation(diverse[k:k + 1]):.1f}', fontsize=9)
    ax.axis('off')
plt.show()

Штраф за похожесть с весом 100 перевесил цель: у трех картинок батча из четырех канал почти не включается, и картинки показывают то, чем можно отличаться друг от друга, а не то, что ищет канал. Вес штрафа — такой же гиперпараметр, как вес штрафа за перепады в ручной версии; его подбирают, глядя на активацию каждой картинки батча.

## 5. Где ломается: сид и размер картинки

Урок подчеркивает: задача невыпуклая, и градиентный подъем приходит в максимум, ближайший к старту. Авторы оригинальной работы видели качественно одну и ту же картинку из девяти стартов; проверим, насколько это верно здесь.

**Задание 3. Получите `lucent_default`-картинку канала `layer4:12` для сидов 0–5. Нарисуйте шесть картинок в ряд с активацией в заголовке и посчитайте среднюю попарную корреляцию пикселей шести картинок. Затем повторите для сида 0 со стороной 64 и 224 вместо 128 (через `param.image(64)` и `param.image(224)`) и сравните картинки трех размеров.**

In [ ]:
# Ваш код здесь

**Задание 4. Средняя корреляция пикселей картинок с разных сидов низкая (около 0.13), а активации близки. Противоречит ли это наблюдению авторов о «качественно одном и том же максимуме»? Что тогда считать ответом метода?**

**Ваш ответ:**

## Что унести из тетради

- **Частотная параметризация — третий прием урока, и в lucent он включен по умолчанию.** Без нее и без преобразований оптимизация снова дает высокочастотный узор с активацией в 107 раз выше лучшего из четырех снимков.
- **Преобразования и параметризация делают разное.** Перепады между пикселями снижает частотная параметризация (с 0.29 до 0.16); преобразования снижают активацию в 3–4 раза и оставляют только узор, устойчивый к сдвигу. Даже с обоими приемами активация в 39 раз выше лучшего снимка: нарисованное лежит вне распределения снимков.
- **Цель и ее веса — тоже выбор.** Канал, нейрон и логит класса дают разные картинки, а штраф за похожесть с весом 100 перевешивает саму цель: три картинки батча из четырех канал не включают.
- **Картинка зависит от сида и размера.** Пиксели разных запусков не совпадают; повторяется текстура, а не композиция.
- **torch-lucent — порт Lucid, последний релиз 0.1.8 вышел в мае 2021.** Здесь он работает с torch 2.11 из Colab, но опирается на kornia 0.4; перед работой проверяйте сигнатуры и совместимость.

Источники:
1. [Olah, Mordvintsev и Schubert, 2017. Feature visualization](https://distill.pub/2017/feature-visualization/)
2. [Erhan et al., 2009. Visualizing higher-layer features of a deep network](https://www.researchgate.net/publication/265022827_Visualizing_Higher-Layer_Features_of_a_Deep_Network)
3. [torch-lucent](https://github.com/greentfrapp/lucent), [Lucid](https://github.com/tensorflow/lucid)